# Validación de normalización de atributos

Este notebook verifica min/max de columnas numéricas en nodos y aristas.
Marca como `fuera_de_rango=True` cualquier columna que no esté en [0,1].

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Ajusta estas rutas si cambian tus archivos
base = Path('/media/vale/KINGSTON/TESIS/data_2026')
nodes_file = base / 'nodes_rib_20260301_0000_to_20260302_0000_enriched_tesis.csv'
edges_file = base / 'edges_rib_20260301_0000_to_20260302_0000_enriched_tesis.csv'

print(f'nodes_file: {nodes_file}')
print(f'edges_file: {edges_file}')
print('nodes existe:', nodes_file.exists())
print('edges existe:', edges_file.exists())

nodes_file: /media/vale/KINGSTON/TESIS/data_2026/nodes_rib_20260301_0000_to_20260302_0000_enriched_tesis.csv
edges_file: /media/vale/KINGSTON/TESIS/data_2026/edges_rib_20260301_0000_to_20260302_0000_enriched_tesis.csv
nodes existe: True
edges existe: True


In [2]:
nodes_df = pd.read_csv(nodes_file)
edges_df = pd.read_csv(edges_file)

print('Nodos shape:', nodes_df.shape)
print('Aristas shape:', edges_df.shape)
print('\nColumnas nodos:', len(nodes_df.columns))
print('Columnas aristas:', len(edges_df.columns))

Nodos shape: (79504, 71)
Aristas shape: (734381, 6)

Columnas nodos: 71
Columnas aristas: 6


In [4]:
def resumen_numerico(df, nombre, excluir=None):
    excluir = set(excluir or [])
    num_cols = [c for c in df.columns if c not in excluir and pd.api.types.is_numeric_dtype(df[c])]
    rows = []
    for c in num_cols:
        s = pd.to_numeric(df[c], errors='coerce')
        mn = float(s.min(skipna=True)) if s.notna().any() else np.nan
        mx = float(s.max(skipna=True)) if s.notna().any() else np.nan
        rows.append({
            'columna': c,
            'min': mn,
            'max': mx,
            'nulos_%': float(s.isna().mean() * 100),
            'unicos': int(s.nunique(dropna=True)),
            'fuera_de_rango': bool((mn < 0) or (mx > 1)) if pd.notna(mn) and pd.notna(mx) else False
        })
    out = pd.DataFrame(rows).sort_values(['fuera_de_rango', 'columna'], ascending=[False, True])
    print(f'\n=== Resumen numérico: {nombre} ===')
    display(out)
    return out

# Excluir IDs/etiquetas que no tienen que estar en [0,1]
excluir_nodos = {'node_id', 'asn'}
excluir_aristas = {'src_id', 'dst_id', 'src_asn', 'dst_asn', 'relationship'}

res_nodes = resumen_numerico(nodes_df, 'NODOS', excluir_nodos)
res_edges = resumen_numerico(edges_df, 'ARISTAS', excluir_aristas)


=== Resumen numérico: NODOS ===


,columna,min,max,nulos_%,unicos,fuera_de_rango
0,weight,1.0,4729694.0,0.0,5903,True
68,PageRank,0.0,1.0,0.0,65397,False
2,fac_count,0.0,1.0,0.0,108,False
3,info_prefixes4,0.0,1.0,0.0,378,False
4,info_prefixes6,0.0,1.0,0.0,227,False
...,...,...,...,...,...,...
11,policy_locations_Preferred,0.0,1.0,0.0,2,False
12,policy_locations_Required - EU,0.0,1.0,0.0,2,False
13,policy_locations_Required - International,0.0,1.0,0.0,2,False
14,policy_locations_Required - US,0.0,1.0,0.0,2,False



=== Resumen numérico: ARISTAS ===


,columna,min,max,nulos_%,unicos,fuera_de_rango
0,weight,1.0,391169.0,0.0,5580,True


In [5]:
# Columnas fuera de [0,1]
bad_nodes = res_nodes[res_nodes['fuera_de_rango']]
bad_edges = res_edges[res_edges['fuera_de_rango']]

print('Columnas de NODOS fuera de [0,1]:', len(bad_nodes))
display(bad_nodes[['columna', 'min', 'max']])

print('Columnas de ARISTAS fuera de [0,1]:', len(bad_edges))
display(bad_edges[['columna', 'min', 'max']])

Columnas de NODOS fuera de [0,1]: 1


,columna,min,max
0,weight,1.0,4729694.0


Columnas de ARISTAS fuera de [0,1]: 1


,columna,min,max
0,weight,1.0,391169.0


In [6]:
# (Opcional) Exportar reportes
out_dir = Path('./results/normalizacion')
out_dir.mkdir(parents=True, exist_ok=True)

res_nodes.to_csv(out_dir / 'reporte_normalizacion_nodos.csv', index=False)
res_edges.to_csv(out_dir / 'reporte_normalizacion_aristas.csv', index=False)

print('Reportes guardados en:', out_dir.resolve())

Reportes guardados en: /home/vale/Escritorio/GIT/TrabajoTesis/results/normalizacion
